# Seasonal Agriculture Performance Analysis

**VOIS AICTE Batch 1 2026–2027 — Major Project**

### Project purpose
Analyze agricultural data across seasons to identify meaningful patterns, trends, relationships and differences in agricultural performance.

This notebook follows the project brief: dataset understanding, data cleaning/preparation, seasonal comparison, pattern/relationship analysis, visualization, interpretation, conclusions and data-driven recommendations.


## 1. Introduction

The dataset contains information about agricultural activities in different seasons and locations. It includes details about crops, weather conditions, farming practices, resource use and financial performance.

For this project, I am mainly focusing on the seasonal side of the data and comparing how agricultural performance changes between seasons.


## 2. Problem Statement

Agricultural performance can change from one season to another because of differences in weather, farming practices, resource availability and market conditions.

The purpose of this project is to study the given data and find out what differences can be seen between seasons. The analysis focuses on yield, profit, water usage, environmental conditions and other useful measures.


## 3. Objectives

The main objectives are:

- Understand the dataset and its columns.
- Check and clean the data before analysis.
- Compare agricultural performance across seasons.
- Look for useful seasonal patterns.
- Study relationships between environmental conditions and agricultural results.
- Compare crops and other groups where useful.
- Create charts that make the results easier to understand.
- Use a statistical test to check whether seasonal differences are meaningful.
- Summarize the main findings and give practical recommendations.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")

print("Shape:", df.shape)
df.head()


In [ ]:
print("Columns:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


## 4. Data Cleaning and Preparation

Before starting the analysis, I checked the size of the dataset, column types, missing values and duplicate rows.

The dataset has 4,000 rows and 28 columns. Missing values are present in rainfall, soil moisture and yield. For rainfall and soil moisture, missing values are filled using the median value of the same season. Yield is an important result variable, so missing yield values are not replaced with made-up values.


In [ ]:
analysis_df = df.copy()

# Seasonal median imputation for environmental variables
for col in ["Rainfall_mm", "Soil_Moisture_pct"]:
    analysis_df[col] = analysis_df.groupby("Season")[col].transform(
        lambda s: s.fillna(s.median())
    )

# Derived metric
analysis_df["Profit_Margin_pct"] = np.where(
    analysis_df["Revenue_INR"] != 0,
    analysis_df["Profit_INR"] / analysis_df["Revenue_INR"] * 100,
    np.nan
)

print("Remaining missing values:")
print(analysis_df.isnull().sum()[analysis_df.isnull().sum() > 0])


## 5. Exploratory Analysis

### 5.1 Dataset composition


In [ ]:
print("Seasons:", analysis_df["Season"].value_counts())
print("\nCrops:", analysis_df["Crop"].value_counts())
print("\nIrrigation methods:", analysis_df["Irrigation_Method"].value_counts())
print("\nStates:", analysis_df["State"].nunique())


## 6. Seasonal Performance Summary

In [ ]:
season_summary = analysis_df.groupby("Season").agg(
    Farms=("Farm_ID", "count"),
    Avg_Yield=("Yield_Tonnes_Ha", "mean"),
    Avg_Production=("Production_Tonnes", "mean"),
    Avg_Revenue=("Revenue_INR", "mean"),
    Avg_Cost=("Total_Cost_INR", "mean"),
    Avg_Profit=("Profit_INR", "mean"),
    Avg_Water=("Water_Used_m3", "mean"),
    Avg_Water_Eff=("Water_Efficiency_t_per_1000m3", "mean"),
    Avg_Risk=("Disease_Pest_Risk_pct", "mean"),
    Avg_Rainfall=("Rainfall_mm", "mean")
).round(2)

season_summary


### 01 Avg Yield By Season

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(season_summary.index, season_summary["Avg_Yield"])
plt.title("Average Yield by Season")
plt.xlabel("Season"); plt.ylabel("Yield (Tonnes/Ha)")
plt.show()

### 02 Avg Profit By Season

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(season_summary.index, season_summary["Avg_Profit"]/1e5)
plt.title("Average Profit by Season")
plt.xlabel("Season"); plt.ylabel("Average Profit (₹ lakh)")
plt.show()

### 03 Avg Water By Season

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(season_summary.index, season_summary["Avg_Water"])
plt.title("Average Water Used by Season")
plt.xlabel("Season"); plt.ylabel("Water Used (m³)")
plt.show()

### 04 Water Eff By Season

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(season_summary.index, season_summary["Avg_Water_Eff"])
plt.title("Average Water Efficiency by Season")
plt.xlabel("Season"); plt.ylabel("Tonnes per 1,000 m³")
plt.show()

### 05 Risk By Season

In [ ]:
plt.figure(figsize=(9,5))
plt.bar(season_summary.index, season_summary["Avg_Risk"])
plt.title("Average Disease/Pest Risk by Season")
plt.xlabel("Season"); plt.ylabel("Risk (%)")
plt.show()

## 7. Crop and Season Analysis

In [ ]:
crop_summary = analysis_df.groupby("Crop").agg(
    Farms=("Farm_ID","count"),
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Avg_Water_Eff=("Water_Efficiency_t_per_1000m3","mean"),
    Avg_Risk=("Disease_Pest_Risk_pct","mean")
).sort_values("Avg_Yield", ascending=False).round(2)

crop_summary


In [ ]:
crop_season = analysis_df.groupby(["Season","Crop"]).agg(
    Avg_Yield=("Yield_Tonnes_Ha","mean"),
    Avg_Profit=("Profit_INR","mean"),
    Farms=("Farm_ID","count")
).round(2)

crop_season.sort_values(["Season","Avg_Yield"], ascending=[True,False]).groupby(level=0).head(3)


In [ ]:
plt.figure(figsize=(9,5))
plt.barh(crop_summary.index[::-1], crop_summary["Avg_Yield"][::-1])
plt.title("Average Yield by Crop")
plt.xlabel("Yield (Tonnes/Ha)")
plt.ylabel("Crop")
plt.show()


## 8. Relationship Analysis

In [ ]:
plt.figure(figsize=(9,5))
plt.scatter(analysis_df["Rainfall_mm"], analysis_df["Yield_Tonnes_Ha"], alpha=0.35)
plt.title("Rainfall vs Yield")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Yield (Tonnes/Ha)")
plt.show()

print("Correlation between rainfall and yield:",
      analysis_df["Rainfall_mm"].corr(analysis_df["Yield_Tonnes_Ha"]).round(3))


In [ ]:
cols = [
    "Rainfall_mm","Avg_Temperature_C","Humidity_pct","Soil_Moisture_pct",
    "Fertilizer_kg_ha","Seed_Quality_Score","Yield_Tonnes_Ha",
    "Revenue_INR","Profit_INR","Water_Used_m3",
    "Water_Efficiency_t_per_1000m3","Disease_Pest_Risk_pct"
]
corr = analysis_df[cols].corr()

plt.figure(figsize=(11,8))
im = plt.imshow(corr, aspect="auto")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.xticks(range(len(cols)), [c[:13] for c in cols], rotation=60, ha="right", fontsize=8)
plt.yticks(range(len(cols)), [c[:16] for c in cols], fontsize=8)
plt.title("Correlation Heatmap of Key Numeric Variables")
for i in range(len(cols)):
    for j in range(len(cols)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=6)
plt.show()


## 9. Statistical Comparison Across Seasons

In [ ]:
yield_groups = [g["Yield_Tonnes_Ha"].dropna() for _, g in analysis_df.groupby("Season")]
profit_groups = [g["Profit_INR"].dropna() for _, g in analysis_df.groupby("Season")]

yield_anova = stats.f_oneway(*yield_groups)
profit_anova = stats.f_oneway(*profit_groups)

print("Yield ANOVA:", yield_anova)
print("Profit ANOVA:", profit_anova)


### Interpretation of the ANOVA results

The ANOVA test is used here to compare the average values between the seasons.

For yield, the p-value is above 0.05, so the data does not give strong evidence that the average yield is different across seasons at the 5% level.

For profit, the p-value is very small, so there is a significant difference in average profit between seasons.

This test shows association/difference in the data. It does not prove that the season itself is the only reason for the difference.


## 10. Key Findings

Based on the dataset:

1. **Kharif has the highest average yield** at approximately **5.64 tonnes/ha**, followed by Rabi (5.08) and Zaid (4.67).
2. **Kharif has the highest average profit** at approximately **₹1.79 lakh**, while Zaid has the lowest average profit.
3. **Kharif also has the highest average rainfall** and the highest average disease/pest risk among the three seasons in this dataset.
4. **Zaid uses the most average water** but has the lowest average water efficiency of the three seasons.
5. **Sugarcane has by far the highest average yield** among crops in the dataset. This large difference should be considered when interpreting overall seasonal yield.
6. Rainfall and disease/pest risk show a positive association in the dataset, but this is an association rather than proof that rainfall causes risk.
7. Profit differs significantly across seasons according to the one-way ANOVA, while the seasonal difference in mean yield is not statistically significant at the 5% level.


## 11. Recommendations

Based on the patterns found in this dataset:

- Seasonal planning should be done using both agricultural and financial measures.
- Zaid deserves further investigation because its average water use is high while its average water efficiency is lower.
- Kharif has the highest average disease/pest risk in this dataset, so this is an area worth monitoring.
- Crop performance should be compared within each season instead of depending only on the overall crop average.
- Rainfall, irrigation, resource use and financial results should be considered together when making decisions.
- More years of data would be useful before applying these findings to real-world planning.


## 12. Conclusion

This project uses data analysis and visualization to compare agricultural performance across seasons. The analysis looked at yield, profit, water use, water efficiency, disease/pest risk and environmental relationships.

The results show that different seasons have different patterns, especially for economic performance and resource use. At the same time, some apparent differences need to be studied along with crop, region and other factors.

Overall, the project shows how a real dataset can be cleaned, analyzed and converted into useful information for seasonal agricultural planning.

## 13. End Users

- Farmers
- Agricultural planners
- Agricultural researchers
- Government/agricultural departments
- Agricultural decision-makers

The findings can support evidence-based seasonal agricultural planning and identify areas requiring further investigation.


## 14. Technology Used

- Python
- Jupyter Notebook
- Pandas
- NumPy
- Matplotlib
- SciPy

The notebook is the main project deliverable for documenting the complete analysis.
